# SAND: every solve folded into the optimiser

SAND (simultaneous analysis and design) hands everything to the optimiser. Its unknowns
are the iteration variables plus every quantity a solve inside the models would
otherwise determine: the copies that open the feedback loops, the coil width from its
root find, and so on. Each of those statements becomes an equality constraint next to
the file's own. There is no inner loop. One SQP runs over one block, each evaluation is
one pass of the models, and the models agree with each other only at the optimum.

SAND takes the same three steps as MDF and IDF and differs in the last. Cut the feedback
loops. State the file's problem. Then `SAND()`: **every** problem on the optimiser's
cycle is `Absorb`ed into it -- its unknowns join the optimiser's, its relations join the
optimiser's, and the problem node goes. The result is what `sand.assemble` builds and
what `session`'s SAND arm solves. This notebook spells the steps out, shows the run
order, and runs it.

In [1]:
import os
import sys
import time
from pathlib import Path

HERE = Path.cwd()                                   # the notebook's own folder
REPO = next(p for p in (HERE, *HERE.parents) if (p / "functional_process").is_dir())
os.chdir(REPO)                                      # input files are named relative to PROCESS/
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import jax

jax.config.update("jax_enable_x64", True)          # PROCESS is float64 throughout
try:                                                # whichever cottax is already on the path
    import cottax
except ImportError:                                 # else the checkout beside this repo
    sys.path.insert(0, str(REPO.parent.parent / "jaxgraph" / "src"))
    import cottax
import numpy as np

print("repo  :", REPO)
print("cottax:", Path(cottax.__file__).parent)

repo  : /home/wrutten/projects/functional_PROCESS/PROCESS
cottax: /home/wrutten/projects/jaxgraph/src/cottax


## The graph

The models of the Helias stellarator input file, as the port declares them. Each node
is one model function; it reads and writes PROCESS's own variables (`.physics.rmajor`
is `data.physics.rmajor`). The file also poses the problem: iteration variables `ixc`,
constraints `icc`, figure of merit. `configurations/stellarator_helias.py` states all
of that as a tree -- the input file converted once -- and `native.reference_of` reads
it without running PROCESS.

In [2]:
CONFIGURATION = "stellarator_helias"                   # configurations/stellarator_helias.py: the machine, its values, its problem

import re

from cottax.interfaces import ExecutableGraph, Plan
from cottax.visualization import problems_at

from functional_process.architecture_examples.notebook_tools import print_recipe
from functional_process.configurations import load
from functional_process.cottax.architectures.evaluate import without_excluded
from functional_process.cottax.input import native
from functional_process.cottax.input.indat import graph_for
from functional_process.cottax.queries import declared
from functional_process.cottax.visualization.grouping import driver_name, problem_kind

configuration = load(CONFIGURATION)
ref = native.reference_of(configuration)      # ixc, icc, bounds, cold values -- PROCESS-free
sw = dict(configuration.problem.switches)     # the static switch values the condition nodes are bound with
machine_graph = graph_for(configuration.machine)
raw = without_excluded(machine_graph)

print(f"{len(raw.nodes)} nodes; cyclic components of sizes {[len(c) for c in raw.graph.cycles]}")
print("problems the models declare themselves:")
for p in declared(raw):
    print(f"   {p.spelling:55s} {problem_kind(raw[p])}")
print(f"\nixc = {ref.ixc}")
print(f"icc = {ref.icc}  ({ref.n_equality} equalities), objective: figure of merit {ref.i_figure_merit}")

152 nodes; cyclic components of sizes [2, 6, 2, 2, 2]
problems the models declare themselves:
   ^problem.stellarator.coils.intersect                    root-find
   ^problem.physics.profiles.ion_vol_avg_temperature       fixed-point
   ^problem.power.delta_eta_step                           fixed-point

ixc = [2, 3, 4, 6, 10, 56, 59, 109]
icc = [2, 16, 24, 8, 17, 18, 67, 82, 83, 62, 32, 34, 35, 65]  (2 equalities), objective: figure of merit 6


## The recipe

### Step 1: cut the feedback loops

Each group of models that feed back into each other is opened with copies. The choice
of which variables to copy is a **scheme**, one graph operation
(`cottax.mdao_architectures`); `mda.SCHEME` is `GaussSeidelMinimal`, the order that
cuts the fewest. Each `FixedPointCut` gives the readers of a variable a copy, `^hat.x`,
and states that the copy equals the computed value -- a fixed-point problem, bound
under `^mda`. The `mda_gauss_seidel` notebook compares the three schemes.

In [3]:
from functional_process.cottax.architectures.mda import SCHEME, cut_graph

plan = Plan(raw) + SCHEME
print_recipe(plan)
for closure in SCHEME.composed(raw):
    print(f"   {closure.problem.spelling}: cut {[c.var.spelling for c in closure.cuts]}")
print("\nsame as mda.cut_graph(raw):", plan.graph == cut_graph(raw))
print("problems the cut minted:", [p.spelling for p in declared(plan.graph) if p not in set(declared(raw))])

   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)

same as mda.cut_graph(raw): True
problems the cut minted: ['^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single']


### Step 2: state the file's problem

Three kinds of node, all stated the way the models are:

- one **constraint** node per active `icc`, owning `^cond.constraints.c<id>`, and one
  for the figure of merit. They compute the same values `models/constraints.py` and
  `models/objectives.py` do;
- one **requirement** beside each constraint, `c = 0` for the equalities and `c <= 0`
  for the inequalities. That, and not the constraint node, is what makes a constraint a
  constraint;
- the **optimiser**: `Optimise(objf, ixc)` at `.Opt`, with no constraints of its own.

A requirement is asserted and answers nothing, so something has to take it on. That is
the architecture's job, in step 3: it `Absorb`s every requirement the design reaches --
the relation joins `.Opt` and the requirement node goes -- and one the design cannot
move is left where it is, for the proof to refuse. `sand.problem_graph` is these two
`Insert`s in one call.

In [4]:
from cottax.interfaces import Insert, Optimise, PathMap, bare_conditions

from functional_process.cottax.architectures import sand
from functional_process.cottax.architectures.sand import (
    OPT,
    condition_nodes,
    iteration_variable_path,
    requirement_nodes,
)
from functional_process.cottax.input.indat import objective_selection
from functional_process.cottax.models.objectives import objective_node, objective_place

nodes, equalities, inequalities, omitted = condition_nodes(plan.graph, ref.icc, ref.n_equality, sw)
name, definition = objective_node(plan.graph.graph.variables, objective_selection(ref.i_figure_merit), sw)
nodes[name] = definition
plan = plan + Insert(PathMap(nodes.items()))                      # one node per constraint, one for the objective

design = tuple(iteration_variable_path(i) for i in ref.ixc)
stated = requirement_nodes(ref.icc, ref.n_equality)               # `c = 0` / `c <= 0` beside each constraint
stated[OPT] = Optimise(objective_place(), design)                 # minimise, over the iteration variables
plan = plan + Insert(PathMap(stated.items()))

requirements = tuple(bare_conditions(plan.graph.definitions))
print(f"{len(requirements)} requirement(s) stated, "
      f"{len(sand.unanswered_requirements(plan.graph))} of them beyond the design's reach")

print_recipe(plan)
if omitted:
    print("constraints this port cannot state yet:", omitted)
reference_graph, _, _ = sand.problem_graph(cut_graph(raw), ref.ixc, ref.icc, ref.n_equality,
                                           ref.i_figure_merit, switch_values=sw)
print("\nsame as sand.problem_graph(...):", plan.graph == reference_graph)

   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)
   insert(.Constraint2, .Constraint16, .Constraint24, .Constraint8, .Constraint17, .Constraint18, .Constraint67,  ...
same as sand.optimise_graph(...): True


Two things are wrong with the graph now, and the proof reports both. Every requirement
is asserted and answers nothing, so no driver can move it. And the optimiser owns the
iteration variables, which almost everything reads, so it closes one big loop over most
of the machine -- a loop that now holds several problems with nothing saying which is
outer. Both are what step 3 decides: which statements the optimiser takes on, and which
are solved inside its iteration. Choosing is choosing the architecture.

In [5]:
from collections import Counter

from cottax.interfaces import violations

big = max(plan.graph.graph.components, key=len)     # the components, with no claim that they can be run
print(f"largest block: {len(big)} of {len(plan.graph.nodes)} nodes, holding",
      [p.spelling for p in declared(plan.graph) if p in set(big)], "\n")

found = list(violations(ExecutableGraph, plan.graph))   # every rule that fails, not just the first
print(Counter(type(v).__name__ for v in found), "\n")
for kind in dict.fromkeys(type(v).__name__ for v in found):   # one example of each
    one = next(v for v in found if type(v).__name__ == kind)
    print(re.sub(r"cycle \(.*?\) declares", "the cycle declares", one.message, flags=re.S), "\n")

largest block: 123 of 170 nodes, holding ['^problem.stellarator.coils.intersect', '^problem.physics.profiles.ion_vol_avg_temperature', '^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single', '.Opt'] 



the block declares several problems ((NodePath(^problem.stellarator.coils.intersect), NodePath(^problem.physics.profiles.ion_vol_avg_temperature), NodePath(^problem.physics.proton_rate_density.cycle), NodePath(^problem.fwbs.f_ster_div_single), NodePath(.Opt))) with nothing saying which is outer -- one driver answers one problem, so `Combine` them into a single problem over every unknown, or `NestInside` one of them. Which is a modelling decision, not something the blocking can read off the graph


### Step 3: what cannot be an SQP unknown

Before absorbing, `sand.assemble` runs the analysis once (`mda_env`) and drops any
consistency statement that is degenerate there -- identically zero, so the equality it
would contribute has no gradient. This file has none, so the check is shown and no
`Delete` is needed. An array-valued unknown is kept: both SQP drivers ravel their
unknowns. The analysis result is kept too -- the solve starts from it.

In [6]:
from functional_process.cottax.architectures.evaluate import mda_env
from functional_process.cottax.architectures.sand import (
    array_valued_problems,
    degenerate_fixed_points,
)

driven, env = mda_env(ref, graph=machine_graph)          # one converged MDA on the cut graph
print("degenerate fixed points (would be dropped):", [p.spelling for p in degenerate_fixed_points(driven, env)] or "none")
print("array-valued statements (kept):            ", [p.spelling for p in array_valued_problems(driven, env)] or "none")

fixed points to drop before folding: none


In [7]:
from cottax.mdao_architectures import SAND

before = set(declared(plan.graph))
plan = plan + SAND()
graph = plan.graph
print("absorbed into the optimiser:", [p.spelling for p in before - set(declared(graph))], "\n")

print("the whole recipe:")
print_recipe(plan)

reference, report = sand.assemble(ref, driven, env, switch_values=sw, drop_arrays=False)
print("\nsame as sand.assemble(...):", graph == reference)
print("problems, in run order:", [p.spelling for p in problems_at(graph) if p is not None])
print(f"the optimiser owns {len(graph[OPT].unknowns)} unknowns and reads {len(graph[OPT].conditions)} conditions")

the whole recipe:
   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)
   insert(.Constraint2, .Constraint16, .Constraint24, .Constraint8, .Constraint17, .Constraint18, .Constraint67,  ...
   residualise(^problem.physics.profiles.ion_vol_avg_temperature)
   residualise(^problem.power.delta_eta_step)
   residualise(^problem.physics.proton_rate_density.cycle)
   residualise(^problem.fwbs.f_ster_div_single)
   combine(^problem.sand <- .Opt, ^problem.stellarator.coils.intersect, ^problem.physics.profiles.ion_vol_avg_tem ...



same as sand.assemble(...): True
problems, in run order: ['^problem.sand']


One problem where there were four. Its unknowns are the eight iteration variables plus
the four the absorbed solves determined. Its conditions are the objective, the file's
fourteen constraints and one equality per absorbed unknown.

### Assign the driver

VMCON on the one problem, attached in the graph by `sand.sand_schedule`.

In [8]:
from functional_process.cottax.architectures.sand import sand_schedule, sand_shape

schedule = sand_schedule(graph, None, bounds=ref.bounds)   # `Assign` VMCON on `.Opt`
shape = sand_shape(schedule)
drive = shape["drive"]
print(f"the SAND block: {shape['drive_nodes']} nodes, {shape['unknowns']} unknowns "
      f"({shape['design']} design), {shape['conditions']} conditions "
      f"({shape['equalities']} equalities, {shape['inequalities']} inequalities); "
      f"{shape['schedule_steps']} schedule steps in all")
print("unknowns:", [u.spelling for u in drive.unknowns])

the SAND block: 124 nodes, 14 unknowns (14 design), 21 conditions (8 equalities, 12 inequalities); 46 schedule steps in all
unknowns: ['.physics.b_plasma_toroidal_on_axis', '.physics.rmajor', '.physics.temp_plasma_electron_vol_avg_kev', '.physics.nd_plasma_electrons_vol_avg', '.physics.hfact', '.tfcoil.t_tf_superconductor_quench', '.tfcoil.f_a_tf_turn_cable_copper', '.physics.f_nd_alpha_thermal_electron', '.stellarator.wp_width_r_min', '.physics.temp_plasma_ion_vol_avg_kev', '.power.delta_eta', '^hat.physics.proton_rate_density', '^hat.physics.fusden_alpha_total', '^hat.fwbs.f_ster_div_single']


## The process

The DSM in run order. One box surrounds the whole coupled block, VMCON on it, and there
is no box inside it. Every former inner solve is now a row of conditions the optimiser
answers. The DSM is written as an interactive page next to this notebook. In the page, hover a
cell for the variables it carries and click a box to fold it.

In [9]:
from functional_process.cottax.visualization.grouping import (
    render_grouped_dsm_html,
    structure_order,
)
from functional_process.cottax.visualization.render_xdsm import SPELLING

drawn = schedule.executable
dsm = render_grouped_dsm_html(
    drawn, order=structure_order(drawn),
    title="stellarator_helias -- SAND: every solve absorbed into one VMCON block",
    file_name="dsm_sand", outdir=str(HERE), write=True, formatter=SPELLING,
)
print("written:", dsm.path)

Using adapted ragraph from debug branch


written: /home/wrutten/projects/functional_PROCESS/PROCESS/functional_process/architecture_examples/sand/dsm_sand.html


## Run it

As `session.solve_block` does it -- and identically for MDF, IDF and SAND, so the three
arms differ in the architecture and in nothing else. The driver records each iterate and
scales each coupling condition by the size of its quantity, so a condition on a value of
order `1e17` is brought to order one. The block starts with the iteration variables at
the input file's values and everything else at a converged analysis of that design. The
schedule then runs step by step, because VMCON runs outside the compiled program.

In [10]:
from functional_process.cottax.architectures import mdf
from functional_process.cottax.architectures.drivers import Status
from functional_process.cottax.architectures.evaluate import (
    inputs_only,
    mda_env,
    run_schedule,
    seed_block,
)
from functional_process.cottax.architectures.mda import seed_starts
from functional_process.cottax.architectures.sand import residual_condition_scales
from functional_process.cottax.architectures.session import (
    SAND_MAX_ITER,
    recorder,
    trace_tail,
)

_driven, env = mda_env(ref, graph=machine_graph)         # one converged MDA on the cut graph, to seed and scale from

trace = []
solve = sand_schedule(graph, None, bounds=ref.bounds,
                      condition_scale=residual_condition_scales(drive, env),
                      callback=recorder(trace), max_iter=SAND_MAX_ITER)
solve_drive = sand_shape(solve)["drive"]
design_vars = set(design)
seeded, borrowed = seed_block(solve, solve_drive, ref.cold, env, design=design_vars)
began = time.perf_counter()
out = run_schedule(solve, inputs_only(solve, seeded), whole=False)
print(f"solved in {time.perf_counter() - began:.1f} s (first solve: includes compilation)")

status = int(np.asarray(mdf.verdict(out, Status, solve_drive)))
iterations, objf, max_eq, min_ie = trace_tail(trace)
print(f"VMCON: status {status}, {iterations} iterations")
print(f"objective {objf:.8f}   max|eq| {max_eq:.1e}   min ineq {min_ie:+.1e}")
print("design:", {i: round(float(np.asarray(out[iteration_variable_path(i)])), 4) for i in ref.ixc})

solved in 8.5 s (first solve: includes compilation)
VMCON: status 0, 43 iterations
objective 1.21844143   max|eq| 1.2e-06   min ineq -4.3e-10
design: {2: 4.7164, 3: 26.6445, 4: 5.7027, 6: 1.7391774970059938e+20, 10: 1.048, 56: 31.8104, 59: 0.7177, 109: 0.0299}



warm solve: 0.92 s, 43 iterations


The same solve is one line: `session.open_session("stellarator_helias").sand()`. `tests/test_architectures.py`
pins it beside the other arms on every regression input file.

In [11]:
RESULT = {"status": status, "iterations": iterations, "objf": objf, "max_eq": max_eq, "min_ie": min_ie,
          "design": {i: float(np.asarray(out[iteration_variable_path(i)])) for i in ref.ixc},
          "unknowns": len(solve_drive.unknowns), "conditions": len(solve_drive.conditions)}
RESULT

{'status': 0,
 'iterations': 43,
 'objf': 1.218441432897411,
 'max_eq': 1.2305890117370283e-06,
 'min_ie': -4.3244252623253487e-10,
 'design': {2: 4.716449671977104,
  3: 26.64445535650798,
  4: 5.702734340773733,
  6: 1.7391774970059938e+20,
  10: 1.047952379234968,
  56: 31.810415391262385,
  59: 0.7176935320205386,
  109: 0.029925036975006085},
 'unknowns': 14,
 'conditions': 21}